# Main Figure v2 — Standalone Panels for Illustrator Assembly

Each panel is saved as an **independent figure** (PDF + PNG) with an
accompanying **source-data TSV**. Assemble in Illustrator.

| Panel | File prefix | Content |
|-------|-------------|---------|
| A1 | `panel_A1_strip_box` | Gene agreement — strip + boxplot |
| A2 | `panel_A2_horiz_bar` | Gene agreement — horizontal bar + strip |
| A3 | `panel_A3_violin` | Gene agreement — violin |
| B1 | `panel_B1_intron_chain_8class` | Transcript concordance by biotype (8-class) |
| B2 | `panel_B2_intron_chain_4group` | Transcript concordance by biotype (4-group) |
| C | `panel_C_jaccard_by_biotype` | Jaccard index distribution |
| D | `panel_D_cds_concordance` | CDS concordance (protein-coding) |
| E | `panel_E_divergence` | GRCh38 divergence categories |

Every `_data.tsv` contains exactly the numbers used to draw the plot.

In [ ]:
import os, shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib import font_manager as fm

# Use local scratch for Matplotlib cache to avoid stale NFS handles
try:
    MPLDIR = Path('/tmp') / f"{os.environ.get('USER','user')}-mplconfig"
    MPLDIR.mkdir(parents=True, exist_ok=True)
    os.environ['MPLCONFIGDIR'] = str(MPLDIR)
    import matplotlib
    ttf_src = Path(matplotlib.get_data_path())/'fonts'/'ttf'
    ttf_dst = MPLDIR/'ttf'
    shutil.copytree(ttf_src, ttf_dst, dirs_exist_ok=True)
    for f in ttf_dst.glob('*.ttf'):
        fm.fontManager.addfont(str(f))
    fm._load_fontmanager(try_read_cache=False)
    mpl.rcParams.update({'svg.fonttype':'none', 'pdf.fonttype':42, 'ps.fonttype':42,
                         'font.family':'DejaVu Sans'})
except Exception as e:
    print('Font init warning:', e)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import gc

warnings.filterwarnings('ignore')

# Publication-quality defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print(f'pandas {pd.__version__}, numpy {np.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')

QC_DIR         = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR    = OUTPUT_DIR / 'results'
SUMMARY_DIR    = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR     = OUTPUT_DIR / 'figures'
INTRON_DIR     = OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
CDS_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'coding_integrity'
DIV_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'divergence'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

def save_panel(fig, prefix):
    """Save a standalone panel as PDF + PNG."""
    fig.savefig(FIGURE_DIR / f'{prefix}.pdf', bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{prefix}.png', dpi=300, bbox_inches='tight')
    print(f'  Saved {prefix}.{{pdf,png}}')

def save_data(df, prefix):
    """Save source data TSV alongside a panel."""
    path = FIGURE_DIR / f'{prefix}_data.tsv'
    df.to_csv(path, sep='\t', index=False)
    print(f'  Saved {prefix}_data.tsv  ({len(df)} rows)')

print(f'Output:  {OUTPUT_DIR}')
print(f'Figures: {FIGURE_DIR}')

In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────

# 8-class intron chain colours (visually grouped)
CLASS_COLORS = {
    'Exact_Match':    '#2a9d8f',
    'Intron_Match':   '#264653',
    'Intron_Subset':  '#457b9d',
    'Intron_Superset':'#74a9cf',
    'Partial_5':      '#e9c46a',
    'Partial_3':      '#f4a261',
    'Other_Partial':  '#e76f51',
    'No_Match':       '#c1121f',
}
CLASS_LABELS = {
    'Exact_Match':     'Exact match',
    'Intron_Match':    'Intron chain match',
    'Intron_Subset':   'Intron subset',
    'Intron_Superset': 'Intron superset',
    'Partial_5':       "Partial (5')",
    'Partial_3':       "Partial (3')",
    'Other_Partial':   'Other partial',
    'No_Match':        'No match',
}
CLASSIFICATION_ORDER = [
    'Exact_Match', 'Intron_Match', 'Intron_Subset', 'Intron_Superset',
    'Partial_5', 'Partial_3', 'Other_Partial', 'No_Match',
]

# 4-group collapsed
GROUP_4_MAP = {
    'Exact_Match':    'Exact match',
    'Intron_Match':   'Same intron chain',
    'Intron_Subset':  'Same intron chain',
    'Intron_Superset':'Same intron chain',
    'Partial_5':      'Partial overlap',
    'Partial_3':      'Partial overlap',
    'Other_Partial':  'Partial overlap',
    'No_Match':       'No match',
}
GROUP_4_COLORS = {
    'Exact match':       '#2a9d8f',
    'Same intron chain': '#457b9d',
    'Partial overlap':   '#f4a261',
    'No match':          '#c1121f',
}
GROUP_4_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match']

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

---
## Load all data

In [ ]:
# ── Gene presence per assembly ────────────────────────────────────────────
funnel_file = SUMMARY_DIR / 'funnel_rung1_gene_presence_per_asm.tsv'
if funnel_file.exists():
    gene_pres = pd.read_csv(funnel_file, sep='\t')
    print(f'Loaded gene presence: {len(gene_pres)} assemblies')
else:
    import re
    ACC_RE = re.compile(r'(GC[AF]_\d+\.\d+)')
    files = sorted(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f'Computing gene presence from {len(files)} files...')
    rows = []
    for f in files:
        m = ACC_RE.search(f.name)
        acc = m.group(1) if m else f.stem
        df = pd.read_csv(f, sep='\t')
        for col in ['present_in_ensembl', 'present_in_cat']:
            df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
        df = df[~df['gene_name'].str.match(r'^ENSG', na=False)]
        n_union = len(df)
        n_both = ((df['present_in_ensembl']) & (df['present_in_cat'])).sum()
        rows.append({'assembly_accession': acc,
                     'n_union_loci': n_union, 'n_both_loci': int(n_both),
                     'pct_gene_presence': n_both / n_union if n_union > 0 else np.nan})
    gene_pres = pd.DataFrame(rows)
    print(f'Computed gene presence: {len(gene_pres)} assemblies')

gene_pres['pct'] = gene_pres['pct_gene_presence'] * 100

# ── Intron chain classification ───────────────────────────────────────────
ic_file = INTRON_DIR / 'intron_chain_by_biotype_per_assembly.tsv'
if ic_file.exists():
    ic_data = pd.read_csv(ic_file, sep='\t')
    print(f'Loaded intron chain: {len(ic_data):,} rows, '
          f'{ic_data["assembly_accession"].nunique()} assemblies')
else:
    print(f'WARNING: {ic_file} not found — run aggregate_intron_chain_by_biotype.py first')
    ic_data = pd.DataFrame()

# ── Jaccard by biotype ────────────────────────────────────────────────────
jac_file = INTRON_DIR / 'jaccard_by_biotype_per_assembly.tsv'
jac_data = pd.read_csv(jac_file, sep='\t') if jac_file.exists() else pd.DataFrame()
if not jac_data.empty:
    print(f'Loaded Jaccard: {len(jac_data):,} rows')

# ── CDS concordance ──────────────────────────────────────────────────────
cds_asm_file = CDS_DIR / 'coding_integrity_per_assembly.tsv'
cds_asm = pd.read_csv(cds_asm_file, sep='\t') if cds_asm_file.exists() else pd.DataFrame()
if not cds_asm.empty:
    print(f'Loaded CDS: {len(cds_asm)} assemblies')

# ── GRCh38 divergence ─────────────────────────────────────────────────────
div_asm_file = DIV_DIR / 'grch38_divergence_per_assembly.tsv'
div_asm = pd.read_csv(div_asm_file, sep='\t') if div_asm_file.exists() else pd.DataFrame()
if not div_asm.empty:
    print(f'Loaded divergence: {div_asm["assembly_accession"].nunique()} assemblies')

---
## Panel A — Gene-level agreement

Three standalone prototypes. All share the same source data TSV.

In [ ]:
# ── Source data ───────────────────────────────────────────────────────────
panel_A_data = gene_pres[['assembly_accession', 'n_union_loci', 'n_both_loci',
                           'pct_gene_presence', 'pct']].copy()
panel_A_data = panel_A_data.rename(columns={'pct': 'pct_gene_presence_x100'})
save_data(panel_A_data, 'panel_A_gene_agreement')

vals = gene_pres['pct'].dropna()
med = vals.median()
print(f'  Median: {med:.1f}%  '
      f'IQR: {vals.quantile(0.25):.1f}\u2013{vals.quantile(0.75):.1f}%  '
      f'n={len(vals)}')

In [ ]:
# ── A1: Strip + Boxplot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(vals, vert=True, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor='#2a9d8f', alpha=0.3),
           medianprops=dict(color='#264653', linewidth=2),
           whiskerprops=dict(color='#264653'),
           capprops=dict(color='#264653'),
           flierprops=dict(marker='o', markersize=3, alpha=0.5))
jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
ax.scatter(np.ones(len(vals)) + jitter, vals, s=6, alpha=0.35,
           color='#2a9d8f', edgecolors='none', zorder=3)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(vals)} assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)
ax.text(1, med + 0.15, f'{med:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A1_strip_box')
plt.show()

In [ ]:
# ── A2: Horizontal bar + dot strip ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.barh(0, med, height=0.5, color='#2a9d8f', alpha=0.85, edgecolor='white')
ax.text(med + 0.1, 0, f'{med:.1f}%', va='center', fontsize=9, fontweight='bold')
jitter_y = np.random.default_rng(42).uniform(-0.18, 0.18, len(vals))
ax.scatter(vals, jitter_y, s=6, alpha=0.4, color='#264653', edgecolors='none', zorder=3)
ax.set_xlabel('Gene loci detected by both methods (%)')
ax.set_yticks([])
ax.set_xlim(max(vals.min() - 1, 90), 101)
ax.axvline(med, color='#264653', linewidth=0.8, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A2_horiz_bar')
plt.show()

In [ ]:
# ── A3: Violin ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 5))
parts = ax.violinplot(vals, vert=True, showmedians=True, showextrema=True)
for pc in parts['bodies']:
    pc.set_facecolor('#2a9d8f'); pc.set_alpha(0.5)
parts['cmedians'].set_color('#264653'); parts['cmedians'].set_linewidth(2)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(vals)} assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)
ax.text(1, med + 0.15, f'{med:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A3_violin')
plt.show()

---
## Panel B — Transcript concordance by biotype

Two standalone figures: B1 (8-class detail) and B2 (4-group collapsed).

In [ ]:
if ic_data.empty:
    print('Skipping Panel B — no intron chain data.')
else:
    # Compute median + IQR per biotype × classification across assemblies
    stats_8 = (
        ic_data.groupby(['biotype', 'classification'])['pct']
        .agg(median_pct='median', q25_pct=lambda x: x.quantile(0.25),
             q75_pct=lambda x: x.quantile(0.75), n_assemblies='count')
        .reset_index()
    )

    # ── Source data for B1 ────────────────────────────────────────────────
    save_data(stats_8, 'panel_B1_intron_chain_8class')

    # Pivot for plotting
    medians_8 = (
        stats_8.pivot(index='biotype', columns='classification', values='median_pct')
        .reindex(index=BIOTYPE_ORDER, columns=CLASSIFICATION_ORDER, fill_value=0)
    )

    # ── Source data for B2 (collapsed) ────────────────────────────────────
    stats_4 = stats_8.copy()
    stats_4['group'] = stats_4['classification'].map(GROUP_4_MAP)
    stats_4_agg = (
        stats_4.groupby(['biotype', 'group'])
        .agg(median_pct=('median_pct', 'sum'),
             q25_pct=('q25_pct', 'sum'),
             q75_pct=('q75_pct', 'sum'))
        .reset_index()
    )
    save_data(stats_4_agg, 'panel_B2_intron_chain_4group')

    medians_4 = (
        stats_4_agg.pivot(index='biotype', columns='group', values='median_pct')
        .reindex(index=BIOTYPE_ORDER, columns=GROUP_4_ORDER, fill_value=0)
    )

    print('\n8-class medians (%):')
    display(medians_8.round(1))
    print('\n4-group medians (%):')
    display(medians_4.round(1))

In [ ]:
if not ic_data.empty:
    # ── B1: Full 8-class ──────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for cls in CLASSIFICATION_ORDER:
        v = medians_8[cls].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=CLASS_COLORS[cls], edgecolor='white', linewidth=0.3,
                label=CLASS_LABELS[cls])
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of gene pairs (median across assemblies)')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_LABELS[c])
               for c in CLASSIFICATION_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B1_intron_chain_8class')
    plt.show()

In [ ]:
if not ic_data.empty:
    # ── B2: Collapsed 4-group ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for grp in GROUP_4_ORDER:
        v = medians_4[grp].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3,
                label=grp)
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of gene pairs (median across assemblies)')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B2_intron_chain_4group')
    plt.show()

---
## Panel C — Jaccard index distribution by biotype

In [ ]:
if jac_data.empty:
    print('Skipping Panel C — no Jaccard data.')
else:
    # ── Source data ────────────────────────────────────────────────────────
    save_data(jac_data, 'panel_C_jaccard_by_biotype')

    # ── Figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))

    bio_stats, bio_labels_plot = [], []
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if bio_df.empty:
            continue
        bio_stats.append({
            'med':    float(bio_df['median'].median()),
            'q1':     float(bio_df['p25'].median()),
            'q3':     float(bio_df['p75'].median()),
            'whislo': float(bio_df['p5'].median()),
            'whishi': float(bio_df['p95'].median()),
            'fliers': [],
        })
        bio_labels_plot.append(BIOTYPE_LABELS[bio])

    positions = np.arange(1, len(bio_stats) + 1)
    bp = ax.bxp(bio_stats, positions=positions, widths=0.55,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor('#2a9d8f'); patch.set_alpha(0.6)

    ax.set_xticks(positions)
    ax.set_xticklabels(bio_labels_plot, rotation=25, ha='right')
    ax.set_ylabel('Jaccard index (best-match exon overlap)')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='grey', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    for i, stats in enumerate(bio_stats):
        ax.text(i + 1, stats['med'] + 0.03, f"{stats['med']:.2f}",
                ha='center', fontsize=8, fontweight='bold')

    plt.tight_layout()
    save_panel(fig, 'panel_C_jaccard_by_biotype')
    plt.show()

---
## Panel D — CDS concordance (protein-coding)

In [ ]:
if cds_asm.empty:
    print('Skipping Panel D — no CDS data.')
else:
    # Find the full-match column
    pct_col = None
    for candidate in ['pct_Full_Match', 'pct_full_match', 'full_match_pct']:
        if candidate in cds_asm.columns:
            pct_col = candidate
            break
    print(f'CDS columns: {list(cds_asm.columns)}')
    print(f'Using: {pct_col}')

    if pct_col:
        # ── Source data ────────────────────────────────────────────────────
        cds_export = cds_asm[['assembly_accession', pct_col]].copy()
        cds_export = cds_export.rename(columns={pct_col: 'pct_cds_full_match'})
        save_data(cds_export, 'panel_D_cds_concordance')

        cds_vals = cds_asm[pct_col].dropna()
        cds_med = cds_vals.median()

        # ── Figure ────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(4, 5))
        ax.boxplot(cds_vals, vert=True, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor='#e76f51', alpha=0.3),
                   medianprops=dict(color='#264653', linewidth=2),
                   whiskerprops=dict(color='#264653'),
                   capprops=dict(color='#264653'),
                   flierprops=dict(marker='o', markersize=3, alpha=0.5))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(cds_vals))
        ax.scatter(np.ones(len(cds_vals)) + jitter, cds_vals, s=6, alpha=0.35,
                   color='#e76f51', edgecolors='none', zorder=3)
        ax.set_ylabel('CDS Full Match (%)')
        ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(cds_vals)} assemblies'])
        ax.text(1, cds_med + 0.15, f'{cds_med:.1f}%', ha='center', fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        plt.tight_layout()
        save_panel(fig, 'panel_D_cds_concordance')
        plt.show()
    else:
        print(f'Could not find full-match column. Available: {list(cds_asm.columns)}')

---
## Panel E — GRCh38 divergence categories

In [ ]:
DIV_COLS   = ['pct_both_agree_reference', 'pct_both_agree_diverged',
              'pct_ensembl_specific', 'pct_cat_specific']
DIV_LABELS = ['Both agree\n(same as ref)', 'Both agree\n(diverged)',
              'Ensembl-\nspecific', 'CAT-\nspecific']
DIV_COLORS = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

if div_asm.empty:
    print('Skipping Panel E — no divergence data.')
else:
    present_cols = [c for c in DIV_COLS if c in div_asm.columns]
    if not present_cols:
        print(f'No divergence columns found. Available: {list(div_asm.columns)[:8]}')
    else:
        # ── Source data ────────────────────────────────────────────────────
        div_export = div_asm[['assembly_accession'] + present_cols].copy()
        save_data(div_export, 'panel_E_divergence')

        # ── Figure ────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 5))
        data = [div_asm[c].dropna().values for c in present_cols]
        labels = [DIV_LABELS[DIV_COLS.index(c)] for c in present_cols]
        colors = [DIV_COLORS[DIV_COLS.index(c)] for c in present_cols]

        parts = ax.violinplot(data, showmedians=True)
        for i, (pc, col) in enumerate(zip(parts['bodies'], colors)):
            pc.set_facecolor(col); pc.set_alpha(0.6)

        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel('Percentage per assembly (%)')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        for i, d in enumerate(data):
            m = np.median(d)
            ax.text(i + 1, m + 1, f'{m:.1f}%', ha='center', fontsize=8, fontweight='bold')

        plt.tight_layout()
        save_panel(fig, 'panel_E_divergence')
        plt.show()

---
## Summary statistics for manuscript text

In [ ]:
print('=' * 60)
print('KEY NUMBERS FOR MANUSCRIPT TEXT')
print('=' * 60)

vals = gene_pres['pct'].dropna()
print(f'\nGene-level agreement:')
print(f'  Median: {vals.median():.1f}%  '
      f'(IQR: {vals.quantile(0.25):.1f}\u2013{vals.quantile(0.75):.1f}%)')
print(f'  Range:  {vals.min():.1f}\u2013{vals.max():.1f}%')
print(f'  N assemblies: {len(vals)}')

if not ic_data.empty:
    print(f'\nTranscript concordance by biotype (median % across assemblies):')
    for bio in BIOTYPE_ORDER:
        if bio in medians_8.index:
            exact = medians_8.loc[bio, 'Exact_Match']
            same_chain = (exact + medians_8.loc[bio, 'Intron_Match']
                          + medians_8.loc[bio, 'Intron_Subset']
                          + medians_8.loc[bio, 'Intron_Superset'])
            no_match = medians_8.loc[bio, 'No_Match']
            print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                  f'Exact={exact:.1f}%, Same chain={same_chain:.1f}%, '
                  f'No match={no_match:.1f}%')

if not jac_data.empty:
    print(f'\nJaccard index (median-of-medians across assemblies):')
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if not bio_df.empty:
            print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                  f'median={bio_df["median"].median():.3f}, '
                  f'IQR={bio_df["p25"].median():.3f}\u2013{bio_df["p75"].median():.3f}')

if not cds_asm.empty and pct_col:
    cv = cds_asm[pct_col].dropna()
    print(f'\nCDS concordance (protein-coding):')
    print(f'  Median: {cv.median():.1f}%  '
          f'(IQR: {cv.quantile(0.25):.1f}\u2013{cv.quantile(0.75):.1f}%)')

print(f'\n{"=" * 60}')
print('Output files written to:', FIGURE_DIR)
print('Done.')